#### 1. Import Libraries

In [102]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import pandas as pd
import numpy as np

#### 2. Create Output Folders

In [103]:
recommendation_path = Path("..//reports//recommendations")
tables_path = Path("..//reports//tables")

recommendation_path.mkdir(
    parents=True,
    exist_ok=True
)

tables_path.mkdir(
    parents=True,
    exist_ok=True
)

print("Folders created successfully.")

Folders created successfully.


#### 3. Load Employee SHAP Dataset

In [104]:
shap_df = pd.read_csv(
    "..//reports//explanations//employee_shap_explanations.csv"
)

raw_df = pd.read_csv("..//data//raw//HR-Employee-Attrition.csv")

processed_df = pd.read_csv(
    "..//data//processed//hr_feature_engineered.csv"
)

In [105]:
raw_df["EmployeeID"] = range(1, len(raw_df) + 1)
processed_df["EmployeeID"] = range(1, len(processed_df) + 1)

In [106]:
df = shap_df.merge(
    raw_df[
        [
            "EmployeeID",
            "Department",
            "JobRole",
            "Gender",
            "Age",
            "BusinessTravel",
            "EducationField",
            "MaritalStatus",
            "MonthlyIncome",
            "YearsAtCompany",
            "OverTime",
            "JobSatisfaction",
            "EnvironmentSatisfaction",
            "WorkLifeBalance"
        ]
    ],
    on="EmployeeID",
    how="left"
)

df = df.merge(
    processed_df[
        [
            "EmployeeID",
            "EngagementScore",
            "WellbeingScore",
            "CareerGrowthScore",
            "LoyaltyScore",
            "AttritionRiskScore"
        ]
    ],
    on="EmployeeID",
    how="left"
)

In [107]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   EmployeeID               1470 non-null   int64  
 1   PredictedProbability     1470 non-null   float64
 2   RiskLevel                1470 non-null   object 
 3   Driver_1                 1470 non-null   object 
 4   SHAP_1                   1470 non-null   float64
 5   FeatureValue_1           1470 non-null   int64  
 6   Impact_1                 1470 non-null   object 
 7   Driver_2                 1470 non-null   object 
 8   SHAP_2                   1470 non-null   float64
 9   FeatureValue_2           1470 non-null   float64
 10  Impact_2                 1470 non-null   object 
 11  Driver_3                 1470 non-null   object 
 12  SHAP_3                   1470 non-null   float64
 13  FeatureValue_3           1470 non-null   float64
 14  Impact_3                

In [108]:
print(f"Rows : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

Rows : 1470
Columns : 33


#### 4. Recommendation Rules

In [109]:
recommendation_rules = {

    "OverTime":{

        "Category":"Work-Life Balance",
        "Recommendation":"Review workload allocation, reduce overtime where possible, and encourage healthy work-life balance."

    },

    "JobSatisfaction":{

        "Category":"Employee Engagement",
        "Recommendation":"Conduct one-on-one meetings, identify concerns, and implement engagement initiatives."

    },

    "EnvironmentSatisfaction":{

        "Category":"Work Environment",
        "Recommendation":"Assess workplace environment and improve employee experience."

    },

    "WorkLifeBalance":{

        "Category":"Work-Life Balance",
        "Recommendation":"Introduce flexible work arrangements and promote employee wellbeing."

    },

    "YearsSinceLastPromotion":{

        "Category":"Career Growth",
        "Recommendation":"Review promotion eligibility and discuss career progression opportunities."

    },

    "MonthlyIncome":{

        "Category":"Compensation",
        "Recommendation":"Benchmark compensation and evaluate salary adjustments where appropriate."

    },

    "DistanceFromHome":{

        "Category":"Flexible Working",
        "Recommendation":"Consider hybrid work options or relocation assistance where feasible."

    },

    "TrainingTimesLastYear":{

        "Category":"Learning & Development",
        "Recommendation":"Provide additional training, mentoring, and professional development opportunities."

    },

    "JobInvolvement":{

        "Category":"Employee Engagement",
        "Recommendation":"Increase employee involvement through meaningful projects and recognition."

    },

    "RelationshipSatisfaction":{

        "Category":"Manager Relationship",
        "Recommendation":"Strengthen manager-employee communication and regular feedback sessions."

    },

    "NumCompaniesWorked":{

        "Category":"Retention",
        "Recommendation":"Understand employee expectations and strengthen long-term retention strategies."

    },

    "YearsAtCompany":{

        "Category":"Retention",
        "Recommendation":"Review career progression and long-term engagement plans."

    }

}

#### 5. Function to Generate AI Recommendation

In [110]:
def generate_recommendation(row):

    recommendations = []
    categories = []

    for i in range(1,4):

        feature = row[f"Driver_{i}"]
        impact = row[f"Impact_{i}"]
        if impact != "Increases Attrition Risk":
            continue

        if feature in recommendation_rules:

            categories.append(
                recommendation_rules[feature]["Category"]
            )

            recommendations.append(
                recommendation_rules[feature]["Recommendation"]
            )

    if len(recommendations)==0:

        return pd.Series([

            "Maintain Engagement",
            "Employee currently demonstrates low attrition risk. Continue regular engagement, recognition, and career development initiatives."

        ])

    return pd.Series([

        ", ".join(sorted(set(categories))),

        " ".join(sorted(set(recommendations)))

    ])

#### 6. Generate AI Recommendations

In [111]:
df[
    [
        "RecommendationCategory",
        "AIRecommendation"
    ]
] = df.apply(
    generate_recommendation,
    axis=1
)

In [112]:
df[
    [
        "EmployeeID",

        "RiskLevel",

        "Driver_1",

        "Driver_2",

        "Driver_3",

        "RecommendationCategory",

        "AIRecommendation"

    ]
].head(10)

,EmployeeID,RiskLevel,Driver_1,Driver_2,Driver_3,RecommendationCategory,AIRecommendation
0,1,Medium,OvertimeRisk,StockOptionLevel,MaritalStatus_Single,Maintain Engagement,Employee currently demonstrates low attrition ...
1,2,Low,OvertimeRisk,FrequentTraveler,NumCompaniesWorked,Retention,Understand employee expectations and strengthe...
2,3,Medium,OvertimeRisk,StockOptionLevel,JobSatisfaction,Maintain Engagement,Employee currently demonstrates low attrition ...
3,4,Low,OvertimeRisk,StockOptionLevel,NumCompaniesWorked,Retention,Understand employee expectations and strengthe...
4,5,Medium,OvertimeRisk,FrequentTraveler,NumCompaniesWorked,Retention,Understand employee expectations and strengthe...
5,6,Medium,OvertimeRisk,StockOptionLevel,JobSatisfaction,Maintain Engagement,Employee currently demonstrates low attrition ...
6,7,Low,OvertimeRisk,DailyRate,Gender_Female,Maintain Engagement,Employee currently demonstrates low attrition ...
7,8,Low,OvertimeRisk,NumCompaniesWorked,DailyRate,Retention,Understand employee expectations and strengthe...
8,9,Low,OvertimeRisk,StockOptionLevel,JobSatisfaction,Maintain Engagement,Employee currently demonstrates low attrition ...
9,10,Low,OvertimeRisk,NumCompaniesWorked,FrequentTraveler,Retention,Understand employee expectations and strengthe...


#### 7. Retention Priority Score

In [113]:
priority = (

    df["PredictedProbability"]*100

).round()

In [114]:
priority += np.where(
    df["RiskLevel"]=="Critical",
    10,
    0
)

priority += np.where(
    df["RiskLevel"]=="High",
    5,
    0
)

priority = priority.clip(
    upper=100
)

In [115]:
df["RetentionPriorityScore"] = priority

In [116]:
df[
    [
        "EmployeeID",
        "RiskLevel",
        "RetentionPriorityScore"

    ]
].sort_values(

    by="RetentionPriorityScore",
    ascending=False

).head(15)

,EmployeeID,RiskLevel,RetentionPriorityScore
1403,1404,High,100.0
838,839,High,100.0
689,690,High,99.0
1060,1061,High,98.0
791,792,High,98.0
829,830,High,96.0
101,102,High,95.0
777,778,High,95.0
1171,1172,High,95.0
573,574,High,95.0


#### 8. Department-Level Summary

In [117]:
department_summary = (
    df.groupby("Department")
      .agg(
          Employees=("EmployeeID", "count"),
          AverageRisk=("PredictedProbability", "mean"),
          AveragePriority=("RetentionPriorityScore", "mean")
      )
      .round(3)
      .reset_index()
)

department_summary

,Department,Employees,AverageRisk,AveragePriority
0,Human Resources,63,0.333,33.746
1,Research & Development,961,0.313,31.659
2,Sales,446,0.427,43.632


#### 9. Risk Distribution by Department

In [118]:
risk_distribution = (
    pd.crosstab(
        df["Department"],
        df["RiskLevel"]
    )
)

risk_distribution

RiskLevel,High,Low,Medium
Department,,,
Human Resources,6,34,23
Research & Development,62,561,338
Sales,87,175,184


#### 10. Recommendation Category Distribution

In [119]:
recommendation_summary = (
    df.groupby("RecommendationCategory")
      .agg(
          Employees=("EmployeeID", "count"),
          AveragePriority=("RetentionPriorityScore", "mean")
      )
      .round(2)
      .sort_values(
          by="Employees",
          ascending=False
      )
      .reset_index()
)

recommendation_summary

,RecommendationCategory,Employees,AveragePriority
0,Retention,884,38.36
1,Maintain Engagement,585,30.84
2,Employee Engagement,1,65.00


#### 11. Executive Summary

In [120]:
executive_summary = []

for department in sorted(df["Department"].unique()):
    dept = df[df["Department"] == department]
    total = len(dept)
    critical = (dept["RiskLevel"] == "Critical").sum()
    high = (dept["RiskLevel"] == "High").sum()
    avg_probability = dept["PredictedProbability"].mean()
    avg_priority = dept["RetentionPriorityScore"].mean()

    top_recommendations = (
        dept["RecommendationCategory"]
            .value_counts()
            .head(3)
            .index
            .tolist()
    )

    executive_summary.append({

        "Department": department,
        "Employees": total,
        "CriticalRiskEmployees": critical,
        "HighRiskEmployees": high,
        "AverageRiskProbability": round(avg_probability,3),
        "AveragePriorityScore": round(avg_priority,1),
        "TopRecommendationCategories":
            ", ".join(top_recommendations)

    })

executive_summary = pd.DataFrame(executive_summary)
executive_summary

,Department,Employees,CriticalRiskEmployees,HighRiskEmployees,AverageRiskProbability,AveragePriorityScore,TopRecommendationCategories
0,Human Resources,63,0,6,0.333,33.7,"Retention, Maintain Engagement"
1,Research & Development,961,0,62,0.313,31.7,"Retention, Maintain Engagement"
2,Sales,446,0,87,0.427,43.6,"Maintain Engagement, Retention, Employee Engag..."


#### 12. Top 25 Employees Requiring Immediate Attention

In [121]:
top_priority = (
    df.sort_values(
        by="RetentionPriorityScore",
        ascending=False
    )
)

top_priority[
    [
        "EmployeeID",
        "Department",
        "JobRole",
        "RiskLevel",
        "RetentionPriorityScore",
        "RecommendationCategory",
        "PredictedProbability"
    ]
].head(25)

,EmployeeID,Department,JobRole,RiskLevel,RetentionPriorityScore,RecommendationCategory,PredictedProbability
1403,1404,Sales,Sales Executive,High,100.0,Maintain Engagement,0.9544
838,839,Sales,Sales Executive,High,100.0,Maintain Engagement,0.9476
689,690,Research & Development,Laboratory Technician,High,99.0,Retention,0.9389
1060,1061,Research & Development,Laboratory Technician,High,98.0,Retention,0.9347
791,792,Sales,Sales Executive,High,98.0,Maintain Engagement,0.9316
829,830,Sales,Sales Executive,High,96.0,Maintain Engagement,0.9067
101,102,Research & Development,Research Scientist,High,95.0,Retention,0.8965
777,778,Research & Development,Laboratory Technician,High,95.0,Retention,0.9023
1171,1172,Research & Development,Laboratory Technician,High,95.0,Retention,0.9017
573,574,Sales,Sales Executive,High,95.0,Maintain Engagement,0.8989


#### 13. Risk Level Summary

In [122]:
risk_summary = (
    df.groupby("RiskLevel")
      .agg(
          Employees=("EmployeeID","count"),
          AverageProbability=("PredictedProbability","mean"),
          AveragePriority=("RetentionPriorityScore","mean")
      )
      .round(3)
      .reset_index()
)

risk_summary

,RiskLevel,Employees,AverageProbability,AveragePriority
0,High,155,0.761,81.052
1,Low,770,0.175,17.539
2,Medium,545,0.476,47.600


#### 14. Top Recommendation Categories by Department

In [123]:
department_recommendations = (
    pd.crosstab(
        df["Department"],
        df["RecommendationCategory"]
    )
)

department_recommendations

RecommendationCategory,Employee Engagement,Maintain Engagement,Retention
Department,,,
Human Resources,0,21,42
Research & Development,0,294,667
Sales,1,270,175


#### 15. Business Intelligence Enrichment

##### 15.1 Attrition Probability (%)

Executives prefer percentages instead of probabilities.

In [124]:
df["AttritionProbability (%)"] = (
    df["PredictedProbability"] * 100
).round(2)

##### 15.2 Global Risk Rank

Rank every employee from highest to lowest risk.

In [125]:
df["GlobalRiskRank"] = (
    df["PredictedProbability"]
    .rank(
        ascending=False,
        method="dense"
    )
    .astype(int)
)

##### 15.3 Department Risk Rank

In [126]:
df["DepartmentRiskRank"] = (
    df.groupby("Department")["PredictedProbability"]
      .rank(
          ascending=False,
          method="dense"
      )
      .astype(int)
)

##### 15.4 Job Role Risk Rank

In [127]:
df["JobRoleRiskRank"] = (
    df.groupby("JobRole")["PredictedProbability"]
      .rank(
          ascending=False,
          method="dense"
      )
      .astype(int)
)

##### 15.5 Age Group

In [128]:
df["AgeGroup"] = pd.cut(

    df["Age"],

    bins=[18,30,40,50,60],

    labels=[
        "18–30",
        "31–40",
        "41–50",
        "51–60"
    ],

    include_lowest=True

)

##### 15.6 Income Band

In [129]:
df["IncomeBand"] = pd.qcut(

    df["MonthlyIncome"],

    q=4,

    labels=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ]

)

##### 15.7 Company Tenure

In [130]:
df["CompanyTenure"] = pd.cut(

    df["YearsAtCompany"],

    bins=[-1,2,5,10,40],

    labels=[
        "New",
        "Early",
        "Experienced",
        "Veteran"
    ]

)

##### 15.8 Retirement Horizon

In [131]:
df["YearsUntilRetirement"] = 60 - df["Age"]

df["YearsUntilRetirement"] = (
    df["YearsUntilRetirement"]
    .clip(lower=0)
)

##### 15.9 Review Priority

In [132]:
conditions = [

    df["RetentionPriorityScore"] >= 90,
    df["RetentionPriorityScore"] >= 75,
    df["RetentionPriorityScore"] >= 50

]

choices = [

    "Immediate",
    "This Month",
    "Quarterly"

]

df["ReviewPriority"] = np.select(

    conditions,
    choices,
    default="Annual"

)

##### 15.10 Executive Action Status

In [133]:
conditions = [

    df["RiskLevel"] == "Critical",
    df["RiskLevel"] == "High",
    df["RiskLevel"] == "Moderate"

]

choices = [

    "Immediate Action",
    "Manager Review",
    "Monitor"

]

df["ExecutiveAction"] = np.select(

    conditions,
    choices,
    default="No Action Required"

)

##### 15.11 Attrition Risk Percentile

In [134]:
df["RiskPercentile"] = (
    df["PredictedProbability"]
      .rank(pct=True)
      .mul(100)
      .round(1)
)

##### 15.12 High-Value Employee Flag

In [135]:
income_cutoff = df["MonthlyIncome"].quantile(0.75)

df["HighValueEmployee"] = np.where(

    (df["MonthlyIncome"] >= income_cutoff) &
    (df["RiskLevel"].isin(["High", "Critical"])),

    "Yes",

    "No"

)

##### 15.13 Retention Action Required

In [136]:
df["RetentionActionRequired"] = np.where(

    df["RiskLevel"].isin(["High", "Critical"]),

    "Yes",

    "No"

)

##### 15.14 Dashboard KPI Summary

In [137]:
print("=" * 50)
print("AI Workforce Dataset Summary")
print("=" * 50)

print(f"Employees: {len(df)}")
print(f"Critical Risk: {(df['RiskLevel']=='Critical').sum()}")
print(f"High Risk: {(df['RiskLevel']=='High').sum()}")
print(f"Immediate Reviews: {(df['ReviewPriority']=='Immediate').sum()}")
print(f"High-Value Employees at Risk: {(df['HighValueEmployee']=='Yes').sum()}")
print(f"Retention Actions Required: {(df['RetentionActionRequired']=='Yes').sum()}")
print("=" * 50)

AI Workforce Dataset Summary
Employees: 1470
Critical Risk: 0
High Risk: 155
Immediate Reviews: 26
High-Value Employees at Risk: 27
Retention Actions Required: 155


##### 15.15 HR Health Score

In [138]:
df["HRHealthScore"] = (

    df["EngagementScore"] * 0.25 +
    df["WellbeingScore"] * 0.20 +
    df["CareerGrowthScore"] * 0.20 +
    df["LoyaltyScore"] * 0.20 +
    (100 - df["AttritionRiskScore"]) * 0.15

).round(1)

##### 15.16 Health Category

In [139]:
conditions = [

    df["HRHealthScore"] >= 85,
    df["HRHealthScore"] >= 70,
    df["HRHealthScore"] >= 55

]

choices = [

    "Excellent",
    "Good",
    "Needs Attention"

]

df["HealthCategory"] = np.select(

    conditions,
    choices,
    default="Critical"

)

##### 15.17 HR Priority Index

In [140]:
df["HRPriorityIndex"] = (

    (100 - df["HRHealthScore"]) * 0.4 +
    df["AttritionProbability (%)"] * 0.6

).round(2)

##### 15.18 HR Priority Level

In [141]:
conditions = [

    df["HRPriorityIndex"] >= 80,
    df["HRPriorityIndex"] >= 60,
    df["HRPriorityIndex"] >= 40

]

choices = [

    "Critical",
    "High",
    "Medium"

]

df["HRPriorityLevel"] = np.select(

    conditions,
    choices,
    default="Low"

)

##### 15.19 Organization Health KPI

In [142]:
print("="*60)
print(f"Average HR Health Score : {df['HRHealthScore'].mean():.1f}")
print(f"Highest Health Score    : {df['HRHealthScore'].max():.1f}")
print(f"Lowest Health Score     : {df['HRHealthScore'].min():.1f}")
print("="*60)

Average HR Health Score : 16.0
Highest Health Score    : 16.6
Lowest Health Score     : 15.1


##### 15.20 Department Health Score

In [143]:
department_health = (

    df.groupby("Department")
      .agg(

          Employees=("EmployeeID","count"),
          AverageHealthScore=("HRHealthScore","mean"),
          AverageRisk=("PredictedProbability","mean"),
          AveragePriority=("HRPriorityIndex","mean")

      )

      .round(2)

      .sort_values(

          by="AverageHealthScore",
          ascending=False

      )

      .reset_index()

)

department_health

,Department,Employees,AverageHealthScore,AverageRisk,AveragePriority
0,Human Resources,63,15.98,0.33,53.58
1,Research & Development,961,15.97,0.31,52.41
2,Sales,446,15.97,0.43,59.21


##### 15.21 Top 20 Healthiest Employees

In [144]:
df.sort_values(

    "HRHealthScore",
    ascending=False

)[

    [

        "EmployeeID",
        "Department",
        "JobRole",
        "HRHealthScore",
        "HealthCategory"

    ]

].head(20)

,EmployeeID,Department,JobRole,HRHealthScore,HealthCategory
193,194,Research & Development,Research Scientist,16.6,Critical
31,32,Research & Development,Healthcare Representative,16.5,Critical
935,936,Sales,Sales Executive,16.5,Critical
1187,1188,Research & Development,Research Scientist,16.5,Critical
289,290,Research & Development,Research Scientist,16.5,Critical
272,273,Research & Development,Research Scientist,16.5,Critical
820,821,Sales,Sales Executive,16.5,Critical
446,447,Sales,Sales Executive,16.5,Critical
982,983,Research & Development,Research Scientist,16.5,Critical
908,909,Sales,Sales Executive,16.5,Critical


##### 15.22 Top 20 Employees Requiring Immediate Intervention

In [145]:
df.sort_values(

    "HRPriorityIndex",
    ascending=False

)[

    [

        "EmployeeID",
        "Department",
        "JobRole",
        "HRPriorityIndex",
        "HRPriorityLevel",
        "RecommendationCategory"

    ]

].head(20)

,EmployeeID,Department,JobRole,HRPriorityIndex,HRPriorityLevel,RecommendationCategory
1403,1404,Sales,Sales Executive,91.02,Critical,Maintain Engagement
838,839,Sales,Sales Executive,90.54,Critical,Maintain Engagement
689,690,Research & Development,Laboratory Technician,90.13,Critical,Retention
1060,1061,Research & Development,Laboratory Technician,89.88,Critical,Retention
791,792,Sales,Sales Executive,89.50,Critical,Maintain Engagement
829,830,Sales,Sales Executive,88.20,Critical,Maintain Engagement
1171,1172,Research & Development,Laboratory Technician,87.98,Critical,Retention
777,778,Research & Development,Laboratory Technician,87.90,Critical,Retention
573,574,Sales,Sales Executive,87.57,Critical,Maintain Engagement
101,102,Research & Development,Research Scientist,87.35,Critical,Retention


In [154]:
priority_employees = df.sort_values(
    "HRPriorityIndex",
    ascending=False
)

In [155]:
priority_employees.to_csv(
    "..//reports//tables//top_intervention_employees.csv",
    index=False
)

#### 16. Save All Tables

In [146]:
department_summary.to_csv(
    "..//reports//tables//department_summary.csv",
    index=False
)

risk_distribution.to_csv(
    "..//reports//tables//risk_distribution.csv"
)

recommendation_summary.to_csv(
    "..//reports//tables//recommendation_summary.csv",
    index=False
)

executive_summary.to_csv(
    "..//reports//tables//executive_summary.csv",
    index=False
)

top_priority.to_csv(
    "..//reports//tables//top_priority_employees.csv",
    index=False
)

risk_summary.to_csv(
    "..//reports//tables//risk_summary.csv",
    index=False
)

department_recommendations.to_csv(
    "..//reports//tables//department_recommendations.csv"
)

department_health.to_csv(
    "..//reports//tables//department_health.csv",
    index=False

)

print("All summary tables saved successfully.")

All summary tables saved successfully.


#### 17. Save Final AI Workforce Dataset

In [147]:
output_path = "..//reports//recommendations//employee_ai_recommendations.csv"

df.to_csv(
    output_path,
    index=False
)

print(f"Dataset saved successfully to:\n{output_path}")

Dataset saved successfully to:
..//reports//recommendations//employee_ai_recommendations.csv


In [148]:
project_summary = pd.DataFrame({

    "Metric": [

        "Total Employees",
        "Departments",
        "Critical Risk Employees",
        "High Risk Employees",
        "Average Attrition Probability",
        "Average Retention Priority"

    ],

    "Value": [

        len(df),

        df["Department"].nunique(),
        (df["RiskLevel"] == "Critical").sum(),
        (df["RiskLevel"] == "High").sum(),
        round(df["PredictedProbability"].mean(), 3),
        round(df["RetentionPriorityScore"].mean(), 2)

    ]

})

project_summary

,Metric,Value
0,Total Employees,1470.000
1,Departments,3.000
2,Critical Risk Employees,0.000
3,High Risk Employees,155.000
4,Average Attrition Probability,0.349
5,Average Retention Priority,35.380


In [149]:
project_summary.to_csv(
    "..//reports//tables//project_summary.csv",
    index=False
)

In [152]:
healthy_employees = df.sort_values(
    "HRHealthScore",
    ascending=False
)

In [153]:
healthy_employees.to_csv(
    "..//reports//tables//top_healthy_employees.csv",
    index=False
)

In [150]:
print("=" * 60)
print("Final AI Recommendation Dataset")
print("=" * 60)

print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

print("\nMissing Values")
print(df.isna().sum().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("=" * 60)

df.head()

Final AI Recommendation Dataset
Rows    : 1470
Columns : 53

Missing Values
0

Duplicate Rows
0


,EmployeeID,PredictedProbability,RiskLevel,Driver_1,SHAP_1,FeatureValue_1,Impact_1,Driver_2,SHAP_2,FeatureValue_2,...,YearsUntilRetirement,ReviewPriority,ExecutiveAction,RiskPercentile,HighValueEmployee,RetentionActionRequired,HRHealthScore,HealthCategory,HRPriorityIndex,HRPriorityLevel
0,1,0.5851,Medium,OvertimeRisk,2.1862,1,Increases Attrition Risk,StockOptionLevel,1.1128,0.0,...,19,Quarterly,No Action Required,83.5,No,No,15.7,Critical,68.83,High
1,2,0.3169,Low,OvertimeRisk,2.0827,0,Increases Attrition Risk,FrequentTraveler,0.9404,1.0,...,11,Annual,No Action Required,50.5,No,No,16.1,Critical,52.57,Medium
2,3,0.4039,Medium,OvertimeRisk,2.3229,1,Increases Attrition Risk,StockOptionLevel,1.3249,0.0,...,23,Annual,No Action Required,62.8,No,No,16.0,Critical,57.83,Medium
3,4,0.1763,Low,OvertimeRisk,2.0261,1,Increases Attrition Risk,StockOptionLevel,1.3411,0.0,...,27,Annual,No Action Required,26.9,No,No,16.3,Critical,44.06,Medium
4,5,0.4298,Medium,OvertimeRisk,1.9883,0,Increases Attrition Risk,FrequentTraveler,0.7550,0.0,...,33,Annual,No Action Required,66.1,No,No,15.7,Critical,59.51,Medium


In [156]:
# All the colums in df
df.columns.tolist()

['EmployeeID',
 'PredictedProbability',
 'RiskLevel',
 'Driver_1',
 'SHAP_1',
 'FeatureValue_1',
 'Impact_1',
 'Driver_2',
 'SHAP_2',
 'FeatureValue_2',
 'Impact_2',
 'Driver_3',
 'SHAP_3',
 'FeatureValue_3',
 'Impact_3',
 'Department',
 'JobRole',
 'Gender',
 'Age',
 'BusinessTravel',
 'EducationField',
 'MaritalStatus',
 'MonthlyIncome',
 'YearsAtCompany',
 'OverTime',
 'JobSatisfaction',
 'EnvironmentSatisfaction',
 'WorkLifeBalance',
 'EngagementScore',
 'WellbeingScore',
 'CareerGrowthScore',
 'LoyaltyScore',
 'AttritionRiskScore',
 'RecommendationCategory',
 'AIRecommendation',
 'RetentionPriorityScore',
 'AttritionProbability (%)',
 'GlobalRiskRank',
 'DepartmentRiskRank',
 'JobRoleRiskRank',
 'AgeGroup',
 'IncomeBand',
 'CompanyTenure',
 'YearsUntilRetirement',
 'ReviewPriority',
 'ExecutiveAction',
 'RiskPercentile',
 'HighValueEmployee',
 'RetentionActionRequired',
 'HRHealthScore',
 'HealthCategory',
 'HRPriorityIndex',
 'HRPriorityLevel']

In [157]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 53 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   EmployeeID                1470 non-null   int64   
 1   PredictedProbability      1470 non-null   float64 
 2   RiskLevel                 1470 non-null   object  
 3   Driver_1                  1470 non-null   object  
 4   SHAP_1                    1470 non-null   float64 
 5   FeatureValue_1            1470 non-null   int64   
 6   Impact_1                  1470 non-null   object  
 7   Driver_2                  1470 non-null   object  
 8   SHAP_2                    1470 non-null   float64 
 9   FeatureValue_2            1470 non-null   float64 
 10  Impact_2                  1470 non-null   object  
 11  Driver_3                  1470 non-null   object  
 12  SHAP_3                    1470 non-null   float64 
 13  FeatureValue_3            1470 non-null   float6

In [158]:
print(df.shape)

(1470, 53)


In [159]:
print(df.isna().sum().sort_values(ascending=False).head(20))

EmployeeID              0
PredictedProbability    0
RiskLevel               0
Driver_1                0
SHAP_1                  0
FeatureValue_1          0
Impact_1                0
Driver_2                0
SHAP_2                  0
FeatureValue_2          0
Impact_2                0
Driver_3                0
SHAP_3                  0
FeatureValue_3          0
Impact_3                0
Department              0
JobRole                 0
Gender                  0
Age                     0
BusinessTravel          0
dtype: int64


In [160]:
print(df.columns.tolist())

['EmployeeID', 'PredictedProbability', 'RiskLevel', 'Driver_1', 'SHAP_1', 'FeatureValue_1', 'Impact_1', 'Driver_2', 'SHAP_2', 'FeatureValue_2', 'Impact_2', 'Driver_3', 'SHAP_3', 'FeatureValue_3', 'Impact_3', 'Department', 'JobRole', 'Gender', 'Age', 'BusinessTravel', 'EducationField', 'MaritalStatus', 'MonthlyIncome', 'YearsAtCompany', 'OverTime', 'JobSatisfaction', 'EnvironmentSatisfaction', 'WorkLifeBalance', 'EngagementScore', 'WellbeingScore', 'CareerGrowthScore', 'LoyaltyScore', 'AttritionRiskScore', 'RecommendationCategory', 'AIRecommendation', 'RetentionPriorityScore', 'AttritionProbability (%)', 'GlobalRiskRank', 'DepartmentRiskRank', 'JobRoleRiskRank', 'AgeGroup', 'IncomeBand', 'CompanyTenure', 'YearsUntilRetirement', 'ReviewPriority', 'ExecutiveAction', 'RiskPercentile', 'HighValueEmployee', 'RetentionActionRequired', 'HRHealthScore', 'HealthCategory', 'HRPriorityIndex', 'HRPriorityLevel']
